# LLM Inference Optimization

LLM inference is fundamentally different from training: it's memory-bandwidth bound during decode, has asymmetric compute phases (prefill vs decode), and costs scale with token count. This note implements cost/latency calculators, batching simulations, and explains speculative decoding.

## What Interviewers Test
- Prefill vs decode phase: why they have different compute characteristics
- Continuous batching vs static batching: why it matters for throughput
- Quantization levels: fp16/int8/int4 quality-memory-speed tradeoffs
- Speculative decoding: how small draft models speed up large model generation
- KV cache size formula and calculator
- Structured/constrained decoding

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# ===== Cost & Latency Calculator =====
def llm_cost_latency(
    prompt_tokens,
    output_tokens,
    model_params_B=7,          # billions
    precision_bytes=2,          # fp16=2, int8=1, int4=0.5
    flops_per_token_forward=2,  # ~2 × params FLOPs per token
    hardware_tflops=312,        # A100 SXM fp16 peak TFLOPS
    hardware_util=0.4,          # typical MFU (model FLOP utilization)
    cost_per_gpu_hour=2.0,      # $/hr
    prefill_batching=True,
):
    """
    Prefill: process entire prompt at once — compute-bound (parallelizable).
    Decode:  generate one token at a time — memory-bandwidth-bound (sequential).
    """
    params = model_params_B * 1e9
    flops_per_param_per_token = 2  # multiply-add = 2 FLOPs
    
    # Prefill: FLOPs = 2 × params × prompt_tokens
    prefill_flops   = flops_per_param_per_token * params * prompt_tokens
    effective_tflops = hardware_tflops * 1e12 * hardware_util
    prefill_time_s   = prefill_flops / effective_tflops

    # Decode: each token requires reading model weights from HBM
    # Memory bandwidth bound: time = (weight bytes + KV cache) / bandwidth
    weight_bytes    = params * precision_bytes
    hbm_bandwidth   = 2e12  # A100 HBM bandwidth ~2 TB/s
    decode_time_per_token = weight_bytes / hbm_bandwidth
    decode_time_s   = decode_time_per_token * output_tokens
    
    total_time_s    = prefill_time_s + decode_time_s
    gpu_hours       = total_time_s / 3600
    cost            = gpu_hours * cost_per_gpu_hour
    
    return {
        'prefill_ms':         prefill_time_s * 1000,
        'decode_ms':          decode_time_s * 1000,
        'decode_ms_per_tok':  decode_time_per_token * 1000,
        'total_ms':           total_time_s * 1000,
        'cost_per_request_usd': cost,
        'throughput_tok_s':   output_tokens / decode_time_s,
    }

print("=== LLaMA-7B (fp16) Inference Estimates ===")
result = llm_cost_latency(prompt_tokens=512, output_tokens=256, model_params_B=7)
for k, v in result.items():
    print(f"  {k:<30}: {v:.4f}")

print()
print("=== LLaMA-7B (int8) Inference Estimates ===")
result_int8 = llm_cost_latency(prompt_tokens=512, output_tokens=256, model_params_B=7, precision_bytes=1)
print(f"  decode_ms_per_tok: {result_int8['decode_ms_per_tok']:.4f}  (vs {result['decode_ms_per_tok']:.4f} fp16 — 2x faster)")
print(f"  throughput_tok_s:  {result_int8['throughput_tok_s']:.1f}  tok/s")


In [ ]:
# ===== Continuous vs Static Batching Simulation =====
import queue
from dataclasses import dataclass, field

@dataclass
class Request:
    req_id: int
    prompt_len: int
    output_len: int
    arrival_time: float

def simulate_static_batching(requests, max_batch=4, token_time_ms=20):
    """Process requests in fixed batches."""
    total_wait = 0
    i = 0
    t = 0
    while i < len(requests):
        batch = requests[i:i+max_batch]
        max_out = max(r.output_len for r in batch)
        # All requests in batch wait for the longest one
        batch_time = max_out * token_time_ms / 1000
        for r in batch:
            wait = max(0, t - r.arrival_time)
            total_wait += wait + (max_out - r.output_len) * token_time_ms / 1000
        t += batch_time
        i += max_batch
    return total_wait / len(requests)

def simulate_continuous_batching(requests, max_batch=4, token_time_ms=20):
    """Continuous batching: replace finished requests immediately."""
    total_wait = 0
    running = []
    waiting = list(requests)
    t = 0
    while running or waiting:
        # Fill batch
        while len(running) < max_batch and waiting:
            req = waiting.pop(0)
            running.append({'req': req, 'tokens_left': req.output_len, 'start': t})
        if not running:
            break
        # Process one decode step
        for r in running:
            r['tokens_left'] -= 1
        t += token_time_ms / 1000
        # Remove finished
        done = [r for r in running if r['tokens_left'] <= 0]
        running = [r for r in running if r['tokens_left'] > 0]
        for d in done:
            total_wait += t - d['req'].arrival_time - d['req'].output_len * token_time_ms / 1000
    return total_wait / len(requests)

np.random.seed(42)
n_req = 20
reqs = [Request(i, np.random.randint(50,200), np.random.randint(10, 100), i*0.1) 
        for i in range(n_req)]

wait_static = simulate_static_batching(reqs)
wait_continuous = simulate_continuous_batching(reqs)
print("=== Batching Strategy Comparison ===")
print(f"Static batching avg wait:     {wait_static:.2f}s")
print(f"Continuous batching avg wait: {wait_continuous:.2f}s")
print(f"Improvement: {wait_static/max(wait_continuous,0.001):.1f}x")
print()
print("Key insight: static batching pads all requests to max length in batch.")
print("Continuous batching replaces finished requests immediately — no padding waste.")


## Quantization Levels

| Level | Memory | Speed | Quality | Notes |
|---|---|---|---|---|
| **fp32** | 4B/param | Baseline | Best | Training only |
| **bf16/fp16** | 2B/param | ~2× fp32 decode | Near-lossless | Standard inference |
| **int8** | 1B/param | ~2× fp16 | <0.5% AUC drop | GPTQ, bitsandbytes |
| **int4** | 0.5B/param | ~2× int8 | 1–3% quality drop | Requires QAT or GPTQ |
| **int2/GGUF** | 0.25B/param | Fast | Significant drop | Edge/local only |

**Rule of thumb for serving budget:** 7B model at int8 fits in a single A100 40GB + room for KV cache. At fp16, you need 14GB + KV cache.


In [ ]:
# ===== KV Cache Size Calculator =====
def kv_cache_size(
    n_layers, n_kv_heads, d_head, max_seq_len, batch_size,
    precision_bytes=2  # fp16
):
    """KV cache size in GB."""
    # 2 × (K + V) × layers × batch × heads × seq × d_head × bytes
    n_elements = 2 * n_layers * batch_size * n_kv_heads * max_seq_len * d_head
    size_bytes  = n_elements * precision_bytes
    return size_bytes / 1e9

# LLaMA-2 7B: 32 layers, 32 heads (no GQA), d_head=128
kv_llama7b = kv_cache_size(n_layers=32, n_kv_heads=32, d_head=128,
                            max_seq_len=4096, batch_size=1)
print(f"LLaMA-2 7B KV cache (4K ctx, batch=1): {kv_llama7b:.2f} GB")

# LLaMA-2 70B uses GQA with n_kv_heads=8 (vs 64 Q heads)
kv_llama70b = kv_cache_size(n_layers=80, n_kv_heads=8, d_head=128,
                             max_seq_len=4096, batch_size=1)
kv_llama70b_nogqa = kv_cache_size(n_layers=80, n_kv_heads=64, d_head=128,
                                   max_seq_len=4096, batch_size=1)
print(f"LLaMA-2 70B KV cache (with GQA 8 heads): {kv_llama70b:.2f} GB")
print(f"LLaMA-2 70B KV cache (without GQA):      {kv_llama70b_nogqa:.2f} GB")
print(f"GQA reduction: {kv_llama70b_nogqa/kv_llama70b:.0f}x")


## Speculative Decoding Concept

Standard decoding: run large model (e.g., 70B) once per token — slow.

Speculative decoding:
1. Small "draft" model generates K tokens cheaply
2. Large "verifier" model checks all K tokens in one batched forward pass
3. Accept tokens up to the first mismatch; reject and resample the rest
4. Average speedup: K × (acceptance rate) — typically 2–3× for γ=4

> 💡 **Interview Tip:** Speculative decoding gives free speedup if a good draft model is available. The key insight is that the verifier's forward pass over K tokens is not K× more expensive than 1 token — due to batching and the KV cache.


## Common Interview Questions

**Q: Why is LLM decode memory-bandwidth-bound rather than compute-bound?**
During decode, you generate one token at a time. For each token, you must read the entire model weights (billions of parameters) from HBM, but perform only a small number of FLOPs (2 × params). Modern GPUs have peak TFLOPS much higher than their memory bandwidth can support, so the bottleneck is memory reads, not computation. Batching helps amortize the weight reads across multiple sequences.

**Q: What is continuous batching and why does it improve throughput?**
Static batching processes a fixed batch until all sequences are done — shorter sequences in the batch waste GPU time waiting for the longest one. Continuous batching replaces a finished sequence immediately with a new one from the queue, keeping GPU utilization high. This alone can improve throughput 5–10× for real workloads with variable output lengths.

**Q: How does speculative decoding work and what are its limitations?**
A small draft model generates K candidate tokens. The large verifier model evaluates all K+1 positions in one forward pass (cheap since it's parallelizable like prefill). Tokens are accepted until the first rejection, then the verifier generates a correction token. Speedup depends on draft acceptance rate — high when draft and verifier agree (common for factual or repetitive text). Fails for highly creative or long-tail outputs where the draft model diverges.

**Q: What is the difference between prefill and decode latency?**
Prefill processes the entire prompt in parallel — it's compute-bound and fast (the whole prompt is one big matrix multiply). Decode generates one token at a time — it's memory-bandwidth-bound and slow per token. For short prompts with long outputs, decode dominates. For RAG with large context, prefill can dominate. This asymmetry is why prompt caching and prefix sharing are valuable.

## Key Takeaways
- Prefill: parallel, compute-bound; Decode: sequential, memory-bandwidth-bound
- Continuous batching: replace finished sequences immediately → 5–10× throughput improvement over static
- Quantization: fp16→int8 = 2× memory reduction, ~2× speed, <0.5% quality; int4 = 4× but 1–3% quality drop
- KV cache: 2 × L × B × H_kv × T × d_head × bytes; GQA reduces by H/H_kv
- Speculative decoding: draft model proposes K tokens; verifier accepts prefix → 2–3× speedup typical
- Memory for 7B fp16: ~14GB weights + KV cache; int8: ~7GB + KV cache